In [1]:
!pip install -U torch torchvision torchaudio transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 4.1 MB/s eta 0:00:000:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 21.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 9.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 28.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "/kaggle/input/models/qnfuioyhgvqpwo/vinasmol-alcapa/pytorch/default/1" 
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto", 
    torch_dtype="auto" 
)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model Size: {total_params / 1e6:.2f} Million Parameters")
if torch.cuda.is_available():
    allocated_memory = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved_memory = torch.cuda.memory_reserved() / (1024 ** 2) 
    
    print(f"Allocated VRAM: {allocated_memory:.2f} MB")
    print(f"Reserved (Cached) VRAM: {reserved_memory:.2f} MB")
else:
    print("CUDA is not available. Model is running on CPU.")

inputs = tokenizer("Xin chào, bạn tên là gì?", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model Size: 368.33 Million Parameters
Allocated VRAM: 346.22 MB
Reserved (Cached) VRAM: 364.00 MB
Xin chào, bạn tên là gì?
T�i tên là Nguyễn Văn Anh.
Bạn có thể cho tôi biết bạn là ai?
T�i là một người đàn ông 30 tuổi, có một con trai 14 tuổi và một con gái 12 tuổi.


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

model.eval()

# Token IDs for A, B, C, and D
def get_letter_tokens(letter):
    ids = tokenizer(letter, add_special_tokens=False)["input_ids"]
    return ids[-1]

vocab_A = get_letter_tokens("A")
vocab_B = get_letter_tokens("B")
vocab_C = get_letter_tokens("C")
vocab_D = get_letter_tokens("D")

print("\n Loading Dataset (Validation Split)...")
dataset = load_dataset("tridm/VMLU", split="validation")

print("\n" + "="*80)
print(" SAMPLE GENERATION & THINKING PROCESS (QUESTION #1)")
print("="*80)

sample = dataset[0]
sample_q = sample["question"]
sample_c = "\n".join(sample["choices"])
sample_prompt = f"Trả lời câu hỏi trắc nghiệm sau bằng cách chọn một trong các đáp án A, B, C hoặc D.\n\nCâu hỏi: {sample_q}\nCác lựa chọn:\n{sample_c}\n\nĐáp án đúng là:"

sample_messages = [{"role": "user", "content": sample_prompt}]
sample_formatted = tokenizer.apply_chat_template(sample_messages, tokenize=False, add_generation_prompt=True)

print(f"PROMPT SENT TO MODEL:\n{sample_prompt}\n")

sample_inputs = tokenizer(sample_formatted, return_tensors="pt").to(model.device)

# Generate a full sentence to see what it "thinks"
print("Generating full creative response...")
with torch.no_grad():
    gen_outputs = model.generate(
        **sample_inputs, 
        max_new_tokens=40,
        pad_token_id=tokenizer.eos_token_id
    )
    
response = tokenizer.decode(gen_outputs[0][sample_inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print(f"\n🧠 MODEL'S ACTUAL GENERATED TEXT:\n\"{response}\"")

print("\n LOGIT PROBABILITIES FOR THIS QUESTION:")
with torch.no_grad():
    output_logits = model(**sample_inputs).logits[0, -1, :]
    print(f"Probability of generating 'A': {output_logits[vocab_A].item():.2f}")
    print(f"Probability of generating 'B': {output_logits[vocab_B].item():.2f}")
    print(f"Probability of generating 'C': {output_logits[vocab_C].item():.2f}")
    print(f"Probability of generating 'D': {output_logits[vocab_D].item():.2f}")
    print(f"Ground Truth Correct Answer: {sample['answer']}")

print("="*80 + "\n")

# Evaluation Loop
correct = 0
total = len(dataset)

print(f"Starting Fast Logit Evaluation on {total} questions...\n")

with torch.no_grad():
    for example in tqdm(dataset):
        question_text = example["question"]
        choices_text = "\n".join(example["choices"])
        
        prompt = f"Trả lời câu hỏi trắc nghiệm sau bằng cách chọn một trong các đáp án A, B, C hoặc D.\n\nCâu hỏi: {question_text}\nCác lựa chọn:\n{choices_text}\n\nĐáp án đúng là:"
        
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)
        
        next_token_logits = outputs.logits[0, -1, :]
        scores = {
            "A": next_token_logits[vocab_A].item(),
            "B": next_token_logits[vocab_B].item(),
            "C": next_token_logits[vocab_C].item(),
            "D": next_token_logits[vocab_D].item(),
        }
        
        predicted_answer = max(scores, key=scores.get)
        actual_answer = example["answer"]
        
        if predicted_answer == actual_answer:
            correct += 1

# Print Final Result
accuracy = (correct / total) * 100
print("\n" + "="*40)
print(f"Final Accuracy: {correct}/{total} ({accuracy:.2f}%)")
print("="*40)



 Loading Dataset (Validation Split)...


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

valid.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

...

Generating validation split:   0%|          | 0/744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9833 [00:00<?, ? examples/s]


 SAMPLE GENERATION & THINKING PROCESS (QUESTION #1)
PROMPT SENT TO MODEL:
Trả lời câu hỏi trắc nghiệm sau bằng cách chọn một trong các đáp án A, B, C hoặc D.

Câu hỏi: Hoạt động nào sau đây của ngân hàng Trung Ương sẽ làm tăng cơ sở tiền tệ
Các lựa chọn:
A. Bán ngoại tệ trên thị trường ngoại hối
B. Cho các ngân hàng thương mại vay
C. Hạ tỷ lệ dự trữ bắt buộc đối với các ngân hàng thương mại
D. Tăng lãi suất chiết khấu

Đáp án đúng là:

Generating full creative response...

🧠 MODEL'S ACTUAL GENERATED TEXT:
"Trong bài viết này, chúng ta sẽ tìm hiểu về các loại thuốc và cách sử dụng chúng.
Thuốc là một loại thuốc được sử dụng để điều trị bệnh hoặc điều trị một vấn đề."

 LOGIT PROBABILITIES FOR THIS QUESTION:
Probability of generating 'A': 7.06
Probability of generating 'B': 7.75
Probability of generating 'C': 7.72
Probability of generating 'D': 7.22
Ground Truth Correct Answer: C

Starting Fast Logit Evaluation on 744 questions...



  0%|          | 0/744 [00:00<?, ?it/s]


Final Accuracy: 170/744 (22.85%)


In [4]:
!pip install rouge-score evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [5]:
import string
import re
import json
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from evaluate import load
from tqdm.auto import tqdm

model.eval()

# Load Rouge Scorer
rouge_scorer = load("rouge")

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the|một|những|các)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def exact_match_score(prediction, ground_truth):
    return (normalize_answer(prediction) == normalize_answer(ground_truth))


squad_file = "/kaggle/input/datasets/nkhachao/vietnamese-squad/dev-v2.0-translated.json"

print(f"\nLoading SQuAD dataset from {squad_file}...")
with open(squad_file, "r", encoding="utf-8") as f:
    squad_data = json.load(f)

eval_examples = []
MAX_EXAMPLES = 200 
for item in squad_data:
    # Each item is a [context, question, answer] list
    if isinstance(item, list) and len(item) == 3:
        context, question, answer = item[0], item[1], item[2]
        if context and question and answer:
            eval_examples.append({
                "context": context,
                "question": question,
                "answer": answer
            })
    if len(eval_examples) >= MAX_EXAMPLES:
        break

print(f"Successfully loaded {len(eval_examples)} questions!")
print(f"Example Question: {eval_examples[0]['question']}")
print(f"Example Answer:   {eval_examples[0]['answer']}")

# EVALUATION LOOP (GENERATION)
total_f1 = 0.0
total_em = 0.0
all_predictions = []
all_references = []

print(f"\nStarting SQuAD Generative Evaluation on {len(eval_examples)} questions...")

for i, example in enumerate(tqdm(eval_examples)):
    context = example["context"]
    question = example["question"]
    truth = example["answer"]
    
    # Prompt explicitly asking the model to extract the short answer
    prompt = f"Dựa vào đoạn văn bản dưới đây, hãy trả lời câu hỏi ngắn gọn nhất có thể.\n\nĐoạn văn: {context}\n\nCâu hỏi: {question}\nTrả lời:"
    
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        gen_outputs = model.generate(
            **inputs, 
            max_new_tokens=30,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1
        )
    
    # Decode only newly generated tokens
    generated_ids = gen_outputs[0][inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Calculate Step-Metrics
    em = exact_match_score(response, truth)
    f1 = f1_score(response, truth)
    
    total_em += int(em)
    total_f1 += f1
    
    all_predictions.append(response)
    all_references.append(truth)
    
    # Print the very first example to verify it's working properly!
    if i == 0:
        print("\n" + "="*80)
        print("🔍 SAMPLE QUALITY CHECK (QUESTION #1)")
        print(f"QUESTION: {question}")
        print(f"TRUTH:    {truth}")
        print(f"PREDICT:  {response}")
        print(f"[EM: {em} | F1: {f1:.2f}]")
        print("="*80 + "\n")

# CALCULATE FINAL DETAILED METRICS
avg_em = (total_em / len(eval_examples)) * 100
avg_f1 = (total_f1 / len(eval_examples)) * 100

rouge_results = rouge_scorer.compute(predictions=all_predictions, references=all_references)

print("\n" + "="*50)
print(f"🇻🇳 SQuAD VIETNAMESE METRICS (N={len(eval_examples)})")
print("="*50)
print(f"Exact Match (EM):     {avg_em:.2f}%")
print(f"Token Overlap (F1):   {avg_f1:.2f}%")
print("-" * 50)
print(f"ROUGE-1:              {rouge_results['rouge1'] * 100:.2f}%")
print(f"ROUGE-2:              {rouge_results['rouge2'] * 100:.2f}%")
print(f"ROUGE-L (Longest Seq):{rouge_results['rougeL'] * 100:.2f}%")
print("="*50)


Loading SQuAD dataset from /kaggle/input/datasets/nkhachao/vietnamese-squad/dev-v2.0-translated.json...
Successfully loaded 200 questions!
Example Question: Normandy nằm ở quốc gia nào?
Example Answer:   Pháp

Starting SQuAD Generative Evaluation on 200 questions...


  0%|          | 0/200 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🔍 SAMPLE QUALITY CHECK (QUESTION #1)
QUESTION: Normandy nằm ở quốc gia nào?
TRUTH:    Pháp
PREDICT:  Trong bài viết này, chúng ta sẽ tìm hiểu về các loại thuốc chống viêm và cách sử dụng chúng.
1. Thuốc chống viêm
Thuốc
[EM: False | F1: 0.00]


🇻🇳 SQuAD VIETNAMESE METRICS (N=200)
Exact Match (EM):     0.00%
Token Overlap (F1):   0.38%
--------------------------------------------------
ROUGE-1:              9.52%
ROUGE-2:              0.71%
ROUGE-L (Longest Seq):7.04%
